In [ ]:
# ==================== 第1格：建立日曆與場站對照 ====================
# 說明：
# 1. 從原始1～6月ZIP取得唯一YouBike場站。
# 2. 日曆直接由日期建立。
# 3. 下載臺灣OpenStreetMap資料並在Colab本機計算300公尺。
# 4. 不使用容易逾時的Overpass即時查詢。
# 5. 分類結果寫入YouBike場站300公尺類型對照.csv。
#
# 本格只建立日曆函式及場站對照。
# 完整原始資料會由第2格處理，不會篩選整點或半點。

%pip -q install scikit-learn requests shapely pyproj

from google.colab import drive
drive.mount('/content/drive')

import hashlib
import json
import re
import shutil
import time
import zipfile
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import requests

from sklearn.neighbors import BallTree
from IPython.display import display
from pyproj import Transformer
from shapely.geometry import Point, shape
from shapely.ops import transform
from shapely.strtree import STRtree


# ==================== A. 路徑設定 ====================

DRIVE_ROOT = Path('/content/drive/MyDrive')
FOLDER_NAME = 'Hackathon_Data'
ZIP_NAME = 'training data_一至六月資料.zip'

WORK_DIR = Path('/content/full_training_work')
OUTPUT_DIR = Path('/content/完整時段_日曆場站結果')

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def find_input_zip():
    direct_path = (
        DRIVE_ROOT /
        FOLDER_NAME /
        ZIP_NAME
    )

    if direct_path.exists():
        return direct_path

    matches = list(
        DRIVE_ROOT.rglob(ZIP_NAME)
    )

    if matches:
        return matches[0]

    raise FileNotFoundError(
        f'找不到{FOLDER_NAME}/{ZIP_NAME}。\n'
        '請確認ZIP檔已放到Google Drive。'
    )


ZIP_PATH = find_input_zip()

print('原始完整資料：', ZIP_PATH)


# ==================== B. 解壓縮原始ZIP ====================

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(WORK_DIR)

CSV_FILES = sorted(
    path
    for path in WORK_DIR.rglob('*.csv')
    if '__MACOSX' not in path.parts
    and not path.name.startswith('._')
)

if not CSV_FILES:
    raise FileNotFoundError(
        'ZIP內找不到CSV檔案。'
    )

print(f'找到{len(CSV_FILES)}個CSV：')

for path in CSV_FILES:
    print(' -', path.relative_to(WORK_DIR))


def detect_encoding(path):
    for encoding in [
        'utf-8-sig',
        'utf-8',
        'cp950',
        'big5',
    ]:
        try:
            pd.read_csv(
                path,
                encoding=encoding,
                nrows=5,
                low_memory=False,
            )
            return encoding
        except UnicodeDecodeError:
            continue

    raise ValueError(
        f'無法判斷CSV編碼：{path.name}'
    )


# ==================== C. 2026年日曆分類 ====================
# 補假日期歸類為「國定假日」。
# 2026年政府行政機關沒有週六補班日。

NATIONAL_HOLIDAYS_2026 = pd.to_datetime([
    '2026-01-01',

    # 春節及補假
    '2026-02-15',
    '2026-02-16',
    '2026-02-17',
    '2026-02-18',
    '2026-02-19',
    '2026-02-20',

    # 和平紀念日及補假
    '2026-02-27',
    '2026-02-28',

    # 兒童節、清明節及補假
    '2026-04-03',
    '2026-04-04',
    '2026-04-05',
    '2026-04-06',

    # 勞動節
    '2026-05-01',

    # 端午節
    '2026-06-19',
]).normalize()

MAKEUP_WORKDAYS_2026 = pd.to_datetime(
    []
).normalize()


def create_calendar_label(datetime_series):
    """產生平日、假日、國定假日、補班日。"""

    dates = datetime_series.dt.normalize()

    labels = pd.Series(
        np.where(
            datetime_series.dt.dayofweek.ge(5),
            '假日',
            '平日',
        ),
        index=datetime_series.index,
        dtype='object',
    )

    labels.loc[
        dates.isin(NATIONAL_HOLIDAYS_2026)
    ] = '國定假日'

    labels.loc[
        dates.isin(MAKEUP_WORKDAYS_2026)
    ] = '補班日'

    return labels


# ==================== D. 取得唯一YouBike場站 ====================

STATION_KEY = [
    '城市',
    '行政區',
    '場站名稱',
]

STATION_SOURCE_COLUMNS = [
    '城市',
    '行政區',
    '場站名稱',
    '經度',
    '緯度',
]

station_parts = []

for file_number, csv_path in enumerate(
    CSV_FILES,
    start=1,
):
    encoding = detect_encoding(csv_path)

    header = pd.read_csv(
        csv_path,
        encoding=encoding,
        nrows=0,
    )

    missing = [
        column
        for column in STATION_SOURCE_COLUMNS
        if column not in header.columns
    ]

    if missing:
        raise ValueError(
            f'{csv_path.name}缺少欄位：{missing}\n'
            f'實際欄位：{list(header.columns)}'
        )

    print(
        f'[{file_number}/{len(CSV_FILES)}] '
        f'取得場站座標：{csv_path.name}'
    )

    for chunk in pd.read_csv(
        csv_path,
        encoding=encoding,
        usecols=STATION_SOURCE_COLUMNS,
        chunksize=300_000,
        low_memory=False,
    ):
        for column in STATION_KEY:
            chunk[column] = (
                chunk[column]
                .fillna('')
                .astype(str)
                .str.strip()
            )

        chunk['城市'] = (
            chunk['城市']
            .replace('', '新北市')
        )

        chunk['經度'] = pd.to_numeric(
            chunk['經度'],
            errors='coerce',
        )

        chunk['緯度'] = pd.to_numeric(
            chunk['緯度'],
            errors='coerce',
        )

        chunk = chunk.dropna(
            subset=['經度', '緯度']
        )

        if chunk.empty:
            continue

        station_parts.append(
            chunk.groupby(
                STATION_KEY,
                as_index=False,
                dropna=False,
            ).agg({
                '經度': 'median',
                '緯度': 'median',
            })
        )


if not station_parts:
    raise RuntimeError(
        '無法從原始CSV取得任何場站座標。'
    )

station_lookup = (
    pd.concat(
        station_parts,
        ignore_index=True,
    )
    .groupby(
        STATION_KEY,
        as_index=False,
        dropna=False,
    )
    .agg({
        '經度': 'median',
        '緯度': 'median',
    })
    .sort_values(STATION_KEY)
    .reset_index(drop=True)
)

print(f'唯一場站數：{len(station_lookup):,}')

display(station_lookup.head())


# ==================== E. 本機OpenStreetMap分類 ====================
# 以下流程會下載臺灣OSM PBF並在Colab本機進行篩選與距離計算。
# 不再呼叫Overpass即時API。

# ==================== F. 安裝Osmium ====================

print('\n安裝Osmium工具……')

subprocess.run(
    [
        'apt-get',
        '-qq',
        'update',
    ],
    check=True,
)

subprocess.run(
    [
        'apt-get',
        '-qq',
        'install',
        '-y',
        'osmium-tool',
    ],
    check=True,
)

version_result = subprocess.run(
    [
        'osmium',
        '--version',
    ],
    check=True,
    capture_output=True,
    text=True,
)

print(
    version_result.stdout
    .splitlines()[0]
)


# ==================== G. 檔案路徑 ====================

OSM_WORK_DIR = Path('/content/osm_local_processing')
OSM_WORK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TAIWAN_PBF = (
    OSM_WORK_DIR /
    'taiwan-latest.osm.pbf'
)

NEW_TAIPEI_PBF = (
    OSM_WORK_DIR /
    'new_taipei_station_area.osm.pbf'
)

POI_PBF = (
    OSM_WORK_DIR /
    'new_taipei_poi.osm.pbf'
)

POI_GEOJSONSEQ = (
    OSM_WORK_DIR /
    'new_taipei_poi.geojsonseq'
)

PBF_URL = (
    'https://download.geofabrik.de/'
    'asia/taiwan-latest.osm.pbf'
)

STATION_MAPPING_FILE = (
    ZIP_PATH.parent /
    'YouBike場站300公尺類型對照.csv'
)


# ==================== H. 檢查本機OSM分類快取 ====================

MAPPING_VERSION = '本機OSM_300公尺_v1'

if STATION_MAPPING_FILE.exists():
    print('\n找到既有的場站對照表，檢查是否為本機OSM版本。')

    saved_mapping = pd.read_csv(
        STATION_MAPPING_FILE,
        encoding='utf-8-sig',
    )

    saved_required = (
        STATION_KEY +
        [
            '經度',
            '緯度',
            '場站',
            '場站分類來源',
            '場站分類狀態',
            '分類版本',
        ]
    )

    missing_saved_columns = [
        column
        for column in saved_required
        if column not in saved_mapping.columns
    ]

    correct_version = (
        not missing_saved_columns
        and saved_mapping['分類版本']
        .fillna('')
        .eq(MAPPING_VERSION)
        .all()
    )

    correct_station_count = (
        len(saved_mapping) == len(station_lookup)
    )

    if correct_version and correct_station_count:
        station_lookup = saved_mapping.copy()
        SKIP_OSM_PROCESSING = True

        print(
            f'快取驗證通過，直接載入：'
            f'{len(station_lookup):,}個場站'
        )

    else:
        SKIP_OSM_PROCESSING = False

        print('既有對照表不是新版，將重新建立並覆蓋。')

        if missing_saved_columns:
            print('缺少欄位：', missing_saved_columns)

else:
    SKIP_OSM_PROCESSING = False

# ==================== I. 下載臺灣OSM資料 ====================

if not SKIP_OSM_PROCESSING:

    if (
        not TAIWAN_PBF.exists()
        or TAIWAN_PBF.stat().st_size < 100_000_000
    ):
        print('\n下載臺灣OpenStreetMap資料。')
        print('檔案約310MB，通常需要數分鐘。')

        subprocess.run(
            [
                'curl',
                '-L',
                '--retry', '8',
                '--retry-delay', '5',
                '--retry-all-errors',
                '--continue-at', '-',
                '--output', str(TAIWAN_PBF),
                PBF_URL,
            ],
            check=True,
        )

    else:
        print('\n使用已下載的臺灣OSM檔：')
        print(TAIWAN_PBF)

    if TAIWAN_PBF.stat().st_size < 100_000_000:
        raise RuntimeError(
            '臺灣OSM檔案大小異常，可能沒有下載完整。'
        )

    print(
        '臺灣OSM檔案大小：'
        f'{TAIWAN_PBF.stat().st_size / 1024**2:,.1f} MB'
    )


# ==================== J. 擷取場站涵蓋範圍 ====================

if not SKIP_OSM_PROCESSING:

    station_lookup['經度'] = pd.to_numeric(
        station_lookup['經度'],
        errors='coerce',
    )

    station_lookup['緯度'] = pd.to_numeric(
        station_lookup['緯度'],
        errors='coerce',
    )

    invalid_coordinates = (
        station_lookup['經度'].isna()
        | station_lookup['緯度'].isna()
        | ~station_lookup['經度'].between(
            119,
            123,
        )
        | ~station_lookup['緯度'].between(
            21,
            26,
        )
    )

    if invalid_coordinates.any():
        display(
            station_lookup.loc[
                invalid_coordinates,
                STATION_KEY +
                ['經度', '緯度']
            ]
        )

        raise ValueError(
            f'有{invalid_coordinates.sum():,}個'
            '場站座標不合理，程式已停止。'
        )

    # 約1公里緩衝範圍
    coordinate_buffer = 0.01

    west = (
        station_lookup['經度'].min()
        - coordinate_buffer
    )

    east = (
        station_lookup['經度'].max()
        + coordinate_buffer
    )

    south = (
        station_lookup['緯度'].min()
        - coordinate_buffer
    )

    north = (
        station_lookup['緯度'].max()
        + coordinate_buffer
    )

    bounding_box = (
        f'{west:.7f},'
        f'{south:.7f},'
        f'{east:.7f},'
        f'{north:.7f}'
    )

    print('\n場站涵蓋範圍：', bounding_box)

    print('從臺灣OSM擷取新北市場站範圍……')

    subprocess.run(
        [
            'osmium',
            'extract',
            '--bbox', bounding_box,
            '--strategy', 'complete_ways',
            '--overwrite',
            '--output', str(NEW_TAIPEI_PBF),
            str(TAIWAN_PBF),
        ],
        check=True,
    )

    print(
        '範圍檔大小：'
        f'{NEW_TAIPEI_PBF.stat().st_size / 1024**2:,.1f} MB'
    )


# ==================== K. 篩選七類設施 ====================

if not SKIP_OSM_PROCESSING:

    osm_filters = [
        # 捷運、火車
        'nwr/railway=station,halt,subway_entrance',
        'nwr/station=subway',
        'nwr/subway=yes',

        # 公車
        'nwr/highway=bus_stop',
        'nwr/amenity=bus_station',
        'nwr/bus=yes',
        'nwr/public_transport=platform,station,stop_position',

        # 學校、商圈、醫院
        (
            'nwr/amenity='
            'school,college,university,kindergarten,'
            'marketplace,hospital,clinic'
        ),

        # 公園
        'nwr/leisure=park,garden,playground',

        # 百貨、購物中心
        'nwr/shop=mall,department_store',
    ]

    print('\n篩選捷運、火車、公車、學校、公園、商圈及醫院……')

    subprocess.run(
        [
            'osmium',
            'tags-filter',
            str(NEW_TAIPEI_PBF),
            *osm_filters,
            '--overwrite',
            '--output', str(POI_PBF),
        ],
        check=True,
    )

    print(
        '設施PBF大小：'
        f'{POI_PBF.stat().st_size / 1024**2:,.1f} MB'
    )

    print('將設施轉成GeoJSON……')

    subprocess.run(
        [
            'osmium',
            'export',
            str(POI_PBF),
            '--output', str(POI_GEOJSONSEQ),
            '--output-format', 'geojsonseq',
            '--overwrite',
        ],
        check=True,
    )


# ==================== L. 設施分類函式 ====================

POI_TYPES = [
    '捷運站',
    '火車站',
    '公車站',
    '學校',
    '公園',
    '商圈',
    '醫院',
]


def classify_osm_properties(properties):
    """將OSM標籤轉成七種場站類型。"""

    railway = str(
        properties.get('railway', '')
    ).lower()

    station = str(
        properties.get('station', '')
    ).lower()

    subway = str(
        properties.get('subway', '')
    ).lower()

    highway = str(
        properties.get('highway', '')
    ).lower()

    amenity = str(
        properties.get('amenity', '')
    ).lower()

    bus = str(
        properties.get('bus', '')
    ).lower()

    public_transport = str(
        properties.get('public_transport', '')
    ).lower()

    leisure = str(
        properties.get('leisure', '')
    ).lower()

    shop = str(
        properties.get('shop', '')
    ).lower()

    name_text = ' '.join([
        str(properties.get('name', '')),
        str(properties.get('name:zh', '')),
        str(properties.get('name:zh-Hant', '')),
        str(properties.get('network', '')),
        str(properties.get('operator', '')),
    ])

    labels = set()

    is_metro = (
        railway == 'subway_entrance'
        or station == 'subway'
        or subway == 'yes'
        or bool(
            re.search(
                r'捷運|輕軌|metro|\bmrt\b',
                name_text,
                flags=re.IGNORECASE,
            )
        )
    )

    is_train = (
        railway in ['station', 'halt']
        and not is_metro
    )

    is_bus = (
        highway == 'bus_stop'
        or amenity == 'bus_station'
        or bus == 'yes'
        or (
            public_transport in [
                'platform',
                'station',
                'stop_position',
            ]
            and bool(
                re.search(
                    r'公車|客運|bus',
                    name_text,
                    flags=re.IGNORECASE,
                )
            )
        )
    )

    if is_metro:
        labels.add('捷運站')

    if is_train:
        labels.add('火車站')

    if is_bus:
        labels.add('公車站')

    if amenity in [
        'school',
        'college',
        'university',
        'kindergarten',
    ]:
        labels.add('學校')

    if leisure in [
        'park',
        'garden',
        'playground',
    ]:
        labels.add('公園')

    if (
        shop in [
            'mall',
            'department_store',
        ]
        or amenity == 'marketplace'
        or bool(
            re.search(
                r'商圈|夜市|黃昏市場|傳統市場|'
                r'百貨|購物中心|商場',
                name_text,
            )
        )
    ):
        labels.add('商圈')

    if amenity in [
        'hospital',
        'clinic',
    ]:
        labels.add('醫院')

    return labels


# ==================== M. 讀取GeoJSON並轉換座標 ====================

if not SKIP_OSM_PROCESSING:

    # WGS84經緯度轉成臺灣公尺座標
    transformer = Transformer.from_crs(
        'EPSG:4326',
        'EPSG:3826',
        always_xy=True,
    )

    geometries_by_type = {
        poi_type: []
        for poi_type in POI_TYPES
    }

    osm_feature_count = 0
    invalid_geometry_count = 0

    with open(
        POI_GEOJSONSEQ,
        'r',
        encoding='utf-8',
    ) as file:

        for line in file:
            line = (
                line
                .strip()
                .lstrip('\x1e')
            )

            if not line:
                continue

            try:
                feature = json.loads(line)

                geometry_json = feature.get(
                    'geometry'
                )

                if not geometry_json:
                    continue

                properties = feature.get(
                    'properties',
                    {},
                )

                labels = classify_osm_properties(
                    properties
                )

                if not labels:
                    continue

                geometry_wgs84 = shape(
                    geometry_json
                )

                if geometry_wgs84.is_empty:
                    continue

                if not geometry_wgs84.is_valid:
                    geometry_wgs84 = (
                        geometry_wgs84.buffer(0)
                    )

                geometry_meter = transform(
                    transformer.transform,
                    geometry_wgs84,
                )

                if geometry_meter.is_empty:
                    continue

                for label in labels:
                    geometries_by_type[
                        label
                    ].append(
                        geometry_meter
                    )

                osm_feature_count += 1

            except Exception:
                invalid_geometry_count += 1

    print(
        f'\n可使用的OSM設施：'
        f'{osm_feature_count:,}個'
    )

    print(
        f'無法解析的幾何資料：'
        f'{invalid_geometry_count:,}個'
    )

    for poi_type in POI_TYPES:
        print(
            f' - {poi_type}：'
            f'{len(geometries_by_type[poi_type]):,}筆'
        )

    if osm_feature_count == 0:
        raise RuntimeError(
            '沒有取得可使用的OSM設施資料。'
        )


# ==================== N. 精確計算300公尺 ====================

if not SKIP_OSM_PROCESSING:

    station_points = [
        Point(
            transformer.transform(
                float(lon),
                float(lat),
            )
        )
        for lon, lat in station_lookup[
            ['經度', '緯度']
        ].itertuples(
            index=False,
            name=None,
        )
    ]

    SEARCH_DISTANCE_METERS = 300

    print('\n開始計算各場站300公尺內設施……')

    for poi_type in POI_TYPES:
        geometries = (
            geometries_by_type[poi_type]
        )

        result_column = (
            f'OSM是否{poi_type}'
        )

        if not geometries:
            station_lookup[
                result_column
            ] = False
            continue

        spatial_tree = STRtree(
            geometries
        )

        nearby_flags = []

        for point in station_points:
            search_area = point.buffer(
                SEARCH_DISTANCE_METERS
            )

            candidates = spatial_tree.query(
                search_area
            )

            found = False

            for candidate in candidates:
                # Shapely 2.x回傳索引
                if isinstance(
                    candidate,
                    (int, np.integer),
                ):
                    candidate_geometry = (
                        geometries[
                            int(candidate)
                        ]
                    )

                # 相容舊版Shapely
                else:
                    candidate_geometry = (
                        candidate
                    )

                if (
                    point.distance(
                        candidate_geometry
                    )
                    <= SEARCH_DISTANCE_METERS
                ):
                    found = True
                    break

            nearby_flags.append(found)

        station_lookup[
            result_column
        ] = nearby_flags

        print(
            f'{poi_type}完成：'
            f'{sum(nearby_flags):,}站'
        )


# ==================== O. 以場站名稱補充明確類型 ====================
# 例如「三峽中山公園」可以合理確認鄰近公園。
# 場站名稱只做保守補充，不會刪除OSM結果。

def classify_by_station_name(station_name):
    text = str(station_name).strip()

    labels = set()

    if re.search(
        r'捷運|輕軌|環狀線',
        text,
    ):
        labels.add('捷運站')

    if (
        re.search(
            r'火車|台鐵|臺鐵|鐵路|車站',
            text,
        )
        and not re.search(
            r'捷運|輕軌',
            text,
        )
    ):
        labels.add('火車站')

    if re.search(
        r'公車|客運|轉運站',
        text,
    ):
        labels.add('公車站')

    if re.search(
        r'國小|國中|高中|高職|大學|學院|'
        r'學校|校區|幼兒園',
        text,
    ):
        labels.add('學校')

    if re.search(
        r'公園|綠地|兒童遊戲場',
        text,
    ):
        labels.add('公園')

    if re.search(
        r'商圈|夜市|黃昏市場|傳統市場|'
        r'市場|百貨|購物中心|商場|老街',
        text,
    ):
        labels.add('商圈')

    if re.search(
        r'醫院|醫療|診所',
        text,
    ):
        labels.add('醫院')

    return labels


if not SKIP_OSM_PROCESSING:

    name_type_sets = (
        station_lookup['場站名稱']
        .apply(classify_by_station_name)
    )

    combined_station_types = []
    classification_sources = []
    classification_statuses = []

    for row_number in range(
        len(station_lookup)
    ):
        osm_types = {
            poi_type
            for poi_type in POI_TYPES
            if bool(
                station_lookup.iloc[
                    row_number
                ][f'OSM是否{poi_type}']
            )
        }

        name_types = (
            name_type_sets.iloc[
                row_number
            ]
        )

        final_types = (
            osm_types |
            name_types
        )

        ordered_types = [
            poi_type
            for poi_type in POI_TYPES
            if poi_type in final_types
        ]

        combined_station_types.append(
            ','.join(ordered_types)
        )

        if osm_types and name_types:
            source = (
                'OpenStreetMap300公尺＋'
                '場站名稱'
            )

        elif osm_types:
            source = (
                'OpenStreetMap300公尺'
            )

        elif name_types:
            source = (
                '場站名稱補充'
            )

        else:
            source = (
                'OpenStreetMap已查詢'
            )

        classification_sources.append(
            source
        )

        if ordered_types:
            status = '已分類'
        else:
            status = (
                '300公尺內無七類設施'
            )

        classification_statuses.append(
            status
        )

    station_lookup['場站'] = (
        combined_station_types
    )

    station_lookup[
        '場站分類來源'
    ] = classification_sources

    station_lookup[
        '場站分類狀態'
    ] = classification_statuses


# ==================== P. 保存最終場站對照 ====================

if not SKIP_OSM_PROCESSING:

    station_lookup['分類版本'] = MAPPING_VERSION

    station_lookup.to_csv(
        STATION_MAPPING_FILE,
        index=False,
        encoding='utf-8-sig',
    )

    print('\n300公尺場站分類完成。')
    print('場站對照表：', STATION_MAPPING_FILE)

    print('\n分類狀態：')

    display(
        station_lookup[
            '場站分類狀態'
        ]
        .value_counts()
        .rename_axis('分類狀態')
        .reset_index(name='場站數')
    )

    print('\n各類型涵蓋站數：')

    type_counts = []

    for poi_type in POI_TYPES:
        count = (
            station_lookup['場站']
            .fillna('')
            .str.split(',')
            .apply(
                lambda values:
                poi_type in values
            )
            .sum()
        )

        type_counts.append({
            '場站類型': poi_type,
            '場站數': int(count),
        })

    display(
        pd.DataFrame(type_counts)
    )

    print('\n場站對照範例：')

    display(
        station_lookup[
            STATION_KEY +
            [
                '經度',
                '緯度',
                '場站',
                '場站分類來源',
                '場站分類狀態',
            ]
        ].head(30)
    )


print(
    '\n第一格處理完成。'
)

print(
    '看到這一行後，就可以執行原本的第2格。'
)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
原始完整資料： /content/drive/MyDrive/Hackathon_Data/training data_一至六月資料.zip
找到6個CSV：
 - training data_一至六月資料/dataset_April.csv
 - training data_一至六月資料/dataset_February.csv
 - training data_一至六月資料/dataset_January.csv
 - training data_一至六月資料/dataset_June.csv
 - training data_一至六月資料/dataset_March.csv
 - training data_一至六月資料/dataset_May.csv
[1/6] 取得場站座標：dataset_April.csv
[2/6] 取得場站座標：dataset_February.csv
[3/6] 取得場站座標：dataset_January.csv
[4/6] 取得場站座標：dataset_June.csv
[5/6] 取得場站座標：dataset_March.csv
[6/6] 取得場站座標：dataset_May.csv
唯一場站數：1,577


,城市,行政區,場站名稱,經度,緯度
0,新北市,三峽區,三峽中山公園,121.36584,24.93561
1,新北市,三峽區,三峽區公所(三峽老街),121.36929,24.93390
2,新北市,三峽區,三峽區礁溪行政中心,121.37592,24.92843
3,新北市,三峽區,三峽原住民族生活文化園區,121.36672,24.94565
4,新北市,三峽區,三峽國中,121.36900,24.93890



安裝Osmium工具……
osmium version 1.16.0

下載臺灣OpenStreetMap資料。
檔案約310MB，通常需要數分鐘。
臺灣OSM檔案大小：311.2 MB

場站涵蓋範圍： 121.3147400,24.8549300,121.9412900,25.3030100
從臺灣OSM擷取新北市場站範圍……
範圍檔大小：77.3 MB

篩選捷運、火車、公車、學校、公園、商圈及醫院……
設施PBF大小：1.7 MB
將設施轉成GeoJSON……

可使用的OSM設施：28,020個
無法解析的幾何資料：0個
 - 捷運站：1,755筆
 - 火車站：44筆
 - 公車站：13,152筆
 - 學校：1,685筆
 - 公園：10,121筆
 - 商圈：744筆
 - 醫院：1,260筆

開始計算各場站300公尺內設施……
捷運站完成：366站
火車站完成：48站
公車站完成：1,523站
學校完成：1,030站
公園完成：1,361站
商圈完成：470站
醫院完成：578站

300公尺場站分類完成。
場站對照表： /content/drive/MyDrive/Hackathon_Data/YouBike場站300公尺類型對照.csv

分類狀態：


,分類狀態,場站數
0,已分類,1569
1,300公尺內無七類設施,8



各類型涵蓋站數：


,場站類型,場站數
0,捷運站,366
1,火車站,49
2,公車站,1523
3,學校,1032
4,公園,1364
5,商圈,479
6,醫院,578



場站對照範例：


,城市,行政區,場站名稱,經度,緯度,場站,場站分類來源,場站分類狀態
0,新北市,三峽區,三峽中山公園,121.36584,24.935610,"公車站,學校,公園,醫院",OpenStreetMap300公尺＋場站名稱,已分類
1,新北市,三峽區,三峽區公所(三峽老街),121.36929,24.933900,"公車站,學校,公園,商圈",OpenStreetMap300公尺＋場站名稱,已分類
2,新北市,三峽區,三峽區礁溪行政中心,121.37592,24.928430,"公車站,學校,公園",OpenStreetMap300公尺,已分類
3,新北市,三峽區,三峽原住民族生活文化園區,121.36672,24.945650,"學校,公園",OpenStreetMap300公尺,已分類
4,新北市,三峽區,三峽國中,121.36900,24.938900,"捷運站,公車站,學校,公園",OpenStreetMap300公尺＋場站名稱,已分類
5,新北市,三峽區,三峽國光青年社會住宅,121.37686,24.937430,"捷運站,公車站,學校,公園",OpenStreetMap300公尺,已分類
6,新北市,三峽區,三峽國小(中山路),121.36779,24.934740,"公車站,學校,公園,商圈",OpenStreetMap300公尺＋場站名稱,已分類
7,新北市,三峽區,三樹大學路口,121.37816,24.939510,"捷運站,公車站,學校,公園",OpenStreetMap300公尺＋場站名稱,已分類
8,新北市,三峽區,三樹大德路口,121.37871,24.941630,"公車站,學校,公園,商圈",OpenStreetMap300公尺,已分類
9,新北市,三峽區,三鶯國民運動中心,121.36858,24.936970,"捷運站,公車站,學校,公園,商圈",OpenStreetMap300公尺,已分類



第一格處理完成。
看到這一行後，就可以執行原本的第2格。


In [ ]:
# ==================== 第2格：完整時段加入日曆、場站並輸出 ====================
# 功能：
# 1. 重新讀取原始1～6月資料。
# 2. 保留整點、半點及所有原始時間，不做篩選或去重複。
# 3. 產生平日、假日、國定假日、補班日。
# 4. 載入第1格建立的300公尺場站分類。
# 5. 欄位依指定順序輸出。
# 6. 輸出六個月份＋中文說明表，共7個CSV。
# 7. 打包ZIP、下載到本機。

from google.colab import drive, files
drive.mount('/content/drive')

import re
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd


# ==================== A. 路徑設定 ====================

DRIVE_ROOT = Path('/content/drive/MyDrive')
HACKATHON_DIR = DRIVE_ROOT / 'Hackathon_Data'

RAW_ZIP_NAME = 'training data_一至六月資料.zip'
STATION_MAPPING_NAME = 'YouBike場站300公尺類型對照.csv'

RAW_ZIP_PATH = HACKATHON_DIR / RAW_ZIP_NAME
STATION_MAPPING_PATH = (
    HACKATHON_DIR /
    STATION_MAPPING_NAME
)

WORK_DIR = Path('/content/full_training_work')
OUTPUT_DIR = Path('/content/完整時段_日曆場站結果')


def find_drive_file(filename):
    """先找Hackathon_Data，找不到再搜尋整個MyDrive。"""
    direct_path = HACKATHON_DIR / filename

    if direct_path.exists():
        return direct_path

    matches = list(
        DRIVE_ROOT.rglob(filename)
    )

    if matches:
        return matches[0]

    return None


RAW_ZIP_PATH = find_drive_file(
    RAW_ZIP_NAME
)

if RAW_ZIP_PATH is None:
    raise FileNotFoundError(
        f'找不到原始資料：\n'
        f'Hackathon_Data/{RAW_ZIP_NAME}'
    )


STATION_MAPPING_PATH = find_drive_file(
    STATION_MAPPING_NAME
)

if STATION_MAPPING_PATH is None:
    raise FileNotFoundError(
        '找不到第1格建立的場站對照表：\n'
        f'Hackathon_Data/{STATION_MAPPING_NAME}\n\n'
        '請先執行第1格，直到畫面顯示：\n'
        '「300公尺場站分類完成」。'
    )


print('原始完整資料：', RAW_ZIP_PATH)
print('場站分類對照：', STATION_MAPPING_PATH)


# ==================== B. 重建暫存資料夾 ====================

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

WORK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ==================== C. 解壓縮原始ZIP ====================

with zipfile.ZipFile(
    RAW_ZIP_PATH,
    'r',
) as zf:
    zf.extractall(WORK_DIR)

CSV_FILES = sorted(
    path
    for path in WORK_DIR.rglob('*.csv')
    if '__MACOSX' not in path.parts
    and not path.name.startswith('._')
)

if not CSV_FILES:
    raise FileNotFoundError(
        '原始ZIP中找不到CSV檔案。'
    )

print(f'\n找到{len(CSV_FILES)}個原始CSV：')

for path in CSV_FILES:
    print(' -', path.relative_to(WORK_DIR))


def detect_encoding(path):
    """自動偵測CSV文字編碼。"""
    for encoding in [
        'utf-8-sig',
        'utf-8',
        'cp950',
        'big5',
    ]:
        try:
            pd.read_csv(
                path,
                encoding=encoding,
                nrows=5,
            )
            return encoding

        except UnicodeDecodeError:
            continue

    raise ValueError(
        f'無法判斷CSV編碼：{path.name}'
    )


# ==================== D. 建立2026年日曆規則 ====================

# 國定假日本日及國定假日補假，
# 在資料中都標記為「國定假日」。
NATIONAL_HOLIDAYS_2026 = pd.to_datetime([
    # 元旦
    '2026-01-01',

    # 小年夜、除夕、春節及補假
    '2026-02-15',
    '2026-02-16',
    '2026-02-17',
    '2026-02-18',
    '2026-02-19',
    '2026-02-20',

    # 和平紀念日補假及和平紀念日
    '2026-02-27',
    '2026-02-28',

    # 兒童節、清明節及補假
    '2026-04-03',
    '2026-04-04',
    '2026-04-05',
    '2026-04-06',

    # 勞動節
    '2026-05-01',

    # 端午節
    '2026-06-19',
]).normalize()

# 2026年政府行政機關沒有補班日
MAKEUP_WORKDAYS_2026 = pd.to_datetime(
    []
).normalize()


def create_calendar_label(datetime_series):
    """
    日曆分類：
    平日、假日、國定假日、補班日。
    """
    dates = datetime_series.dt.normalize()

    labels = pd.Series(
        np.where(
            datetime_series.dt.dayofweek.ge(5),
            '假日',
            '平日',
        ),
        index=datetime_series.index,
        dtype='object',
    )

    # 國定假日優先於一般週末
    labels.loc[
        dates.isin(
            NATIONAL_HOLIDAYS_2026
        )
    ] = '國定假日'

    # 若未來補充補班日，補班日優先
    labels.loc[
        dates.isin(
            MAKEUP_WORKDAYS_2026
        )
    ] = '補班日'

    return labels


# ==================== E. 讀取場站300公尺對照表 ====================

STATION_KEY = [
    '城市',
    '行政區',
    '場站名稱',
]

POI_TYPES = [
    '捷運站',
    '火車站',
    '公車站',
    '學校',
    '公園',
    '商圈',
    '醫院',
]

station_lookup = pd.read_csv(
    STATION_MAPPING_PATH,
    encoding='utf-8-sig',
)

required_mapping_columns = (
    STATION_KEY
    + ['場站']
)

missing_mapping_columns = [
    column
    for column in required_mapping_columns
    if column not in station_lookup.columns
]

if missing_mapping_columns:
    raise ValueError(
        '場站對照表缺少必要欄位：'
        f'{missing_mapping_columns}\n'
        f'實際欄位：{list(station_lookup.columns)}'
    )


def clean_station_labels(value):
    """整理中文多重場站標記並移除重複。"""
    if pd.isna(value):
        return ''

    parts = re.split(
        r'[,，、|;/；\s]+',
        str(value).strip(),
    )

    found = set(
        part.strip()
        for part in parts
        if part.strip()
    )

    # 固定依指定的七種類型排序
    return ','.join(
        label
        for label in POI_TYPES
        if label in found
    )


def combine_station_labels(series):
    """同一場站若出現多筆對照，合併所有類型。"""
    found = set()

    for value in series:
        cleaned = clean_station_labels(
            value
        )

        for label in cleaned.split(','):
            if label:
                found.add(label)

    return ','.join(
        label
        for label in POI_TYPES
        if label in found
    )


for column in STATION_KEY:
    station_lookup[column] = (
        station_lookup[column]
        .fillna('')
        .astype(str)
        .str.strip()
    )

station_lookup['城市'] = (
    station_lookup['城市']
    .replace('', '新北市')
)

station_lookup['場站'] = (
    station_lookup['場站']
    .apply(clean_station_labels)
)

# 確保每個場站只有一筆，避免many_to_one合併失敗
station_merge_table = (
    station_lookup
    .groupby(
        STATION_KEY,
        as_index=False,
    )
    .agg({
        '場站': combine_station_labels
    })
)

print(
    f'\n成功載入場站對照：'
    f'{len(station_merge_table):,}站'
)

print('有場站分類：', end=' ')

print(
    f"{station_merge_table['場站'].ne('').sum():,}站"
)

print('300公尺內無指定類型：', end=' ')

print(
    f"{station_merge_table['場站'].eq('').sum():,}站"
)

display(
    station_merge_table.head(20)
)


# ==================== F. 最終欄位順序 ====================

FINAL_COLUMNS = [
    '日期',
    '城市',
    '行政區',
    '場站名稱',
    '總車柱數',
    '可借車數',
    '可還位數',
    '經度',
    '緯度',
    '日曆',
    '場站',
    '資料品質標記',
    '異常原因',
]

REQUIRED_COLUMNS = [
    '日期',
    '城市',
    '行政區',
    '場站名稱',
    '總車柱數',
    '可借車數',
    '可還位數',
    '經度',
    '緯度',
]


# ==================== G. 月份輸出檔設定 ====================

monthly_paths = {
    month: (
        OUTPUT_DIR /
        f'training_data_{month:02d}月_'
        f'完整時段_日曆場站中文版.csv'
    )
    for month in range(1, 7)
}

month_counts = {
    month: 0
    for month in range(1, 7)
}

minute_counts = {}

source_total_rows = 0
output_total_rows = 0
unclassified_rows = 0
unclassified_stations = set()


# ==================== H. 分批處理完整原始資料 ====================
# 沒有整點篩選、沒有floor、沒有round、沒有drop_duplicates。

for file_number, csv_path in enumerate(
    CSV_FILES,
    start=1,
):
    encoding = detect_encoding(
        csv_path
    )

    header = pd.read_csv(
        csv_path,
        encoding=encoding,
        nrows=0,
    )

    missing_columns = [
        column
        for column in REQUIRED_COLUMNS
        if column not in header.columns
    ]

    if missing_columns:
        raise ValueError(
            f'{csv_path.name}缺少欄位：'
            f'{missing_columns}\n'
            f'實際欄位：{list(header.columns)}'
        )

    print(
        f'\n[{file_number}/{len(CSV_FILES)}] '
        f'處理：{csv_path.name}'
    )

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            csv_path,
            encoding=encoding,
            chunksize=300_000,
            low_memory=False,
        ),
        start=1,
    ):
        source_total_rows += len(chunk)

        # 如果原始資料已有舊欄位，先移除後重新建立
        chunk = chunk.drop(
            columns=[
                '日曆',
                '場站',
            ],
            errors='ignore',
        )

        if '資料品質標記' not in chunk.columns:
            chunk['資料品質標記'] = ''

        if '異常原因' not in chunk.columns:
            chunk['異常原因'] = ''

        try:
            chunk['日期'] = pd.to_datetime(
                chunk['日期'],
                errors='coerce',
                format='mixed',
            )

        except TypeError:
            chunk['日期'] = pd.to_datetime(
                chunk['日期'],
                errors='coerce',
            )

        bad_dates = int(
            chunk['日期'].isna().sum()
        )

        if bad_dates:
            raise ValueError(
                f'{csv_path.name}第'
                f'{chunk_number}批有'
                f'{bad_dates:,}筆日期無法解析。\n'
                '為避免原始資料遺失，'
                '程式已停止。'
            )

        invalid_months = (
            ~chunk['日期']
            .dt.month
            .isin(range(1, 7))
        )

        if invalid_months.any():
            invalid_examples = (
                chunk.loc[
                    invalid_months,
                    '日期'
                ]
                .head(10)
                .tolist()
            )

            raise ValueError(
                '原始資料出現1～6月以外日期：'
                f'{invalid_examples}'
            )

        # 統計分鐘，確認00分與30分均有保留
        chunk_minute_counts = (
            chunk['日期']
            .dt.minute
            .value_counts()
        )

        for minute, count in (
            chunk_minute_counts.items()
        ):
            minute_counts[int(minute)] = (
                minute_counts.get(
                    int(minute),
                    0,
                )
                + int(count)
            )

        for column in STATION_KEY:
            chunk[column] = (
                chunk[column]
                .fillna('')
                .astype(str)
                .str.strip()
            )

        chunk['城市'] = (
            chunk['城市']
            .replace('', '新北市')
        )

        # 建立中文日曆欄位
        chunk['日曆'] = (
            create_calendar_label(
                chunk['日期']
            )
        )

        # 加入300公尺場站分類
        chunk = chunk.merge(
            station_merge_table,
            on=STATION_KEY,
            how='left',
            validate='many_to_one',
        )

        chunk['場站'] = (
            chunk['場站']
            .fillna('')
            .astype(str)
        )

        no_station_type = (
            chunk['場站']
            .str.strip()
            .eq('')
        )

        unclassified_rows += int(
            no_station_type.sum()
        )

        unclassified_names = (
            chunk.loc[
                no_station_type,
                '場站名稱'
            ]
            .drop_duplicates()
            .tolist()
        )

        unclassified_stations.update(
            unclassified_names
        )

        # 嚴格依使用者指定順序
        chunk = chunk[
            FINAL_COLUMNS
        ].copy()

        # 每個月份分開追加寫入
        for month in range(1, 7):
            month_data = chunk.loc[
                chunk['日期']
                .dt.month
                .eq(month)
            ].copy()

            if month_data.empty:
                continue

            # 每一批內依日期與場站排序
            month_data = (
                month_data
                .sort_values(
                    [
                        '日期',
                        '城市',
                        '行政區',
                        '場站名稱',
                    ],
                    kind='mergesort',
                )
            )

            output_path = (
                monthly_paths[month]
            )

            month_data.to_csv(
                output_path,
                mode='a',
                header=not output_path.exists(),
                index=False,
                encoding='utf-8-sig',
            )

            written_rows = len(
                month_data
            )

            month_counts[month] += (
                written_rows
            )

            output_total_rows += (
                written_rows
            )

        print(
            f'  第{chunk_number}批完成：'
            f'{len(chunk):,}筆'
        )


# ==================== I. 中文標記說明表 ====================

label_table = pd.DataFrame([
    [
        '日曆',
        '平日',
        '星期一至星期五，且不是國定假日或補班日',
    ],
    [
        '日曆',
        '假日',
        '一般星期六或星期日',
    ],
    [
        '日曆',
        '國定假日',
        '國定假日及國定假日補假日期',
    ],
    [
        '日曆',
        '補班日',
        '政府公告的補行上班日；2026年無補班日',
    ],
    [
        '場站',
        '捷運站',
        'YouBike場站300公尺內有捷運站或捷運出入口',
    ],
    [
        '場站',
        '火車站',
        'YouBike場站300公尺內有火車站',
    ],
    [
        '場站',
        '公車站',
        'YouBike場站300公尺內有公車站',
    ],
    [
        '場站',
        '學校',
        'YouBike場站300公尺內有學校、大學、學院或幼兒園',
    ],
    [
        '場站',
        '公園',
        'YouBike場站300公尺內有公園、花園或遊樂場',
    ],
    [
        '場站',
        '商圈',
        'YouBike場站300公尺內有夜市、市場、百貨或購物中心',
    ],
    [
        '場站',
        '醫院',
        'YouBike場站300公尺內有醫院或診所',
    ],
], columns=[
    '欄位',
    '中文標記',
    '說明',
])

LABEL_FILE = (
    OUTPUT_DIR /
    '日曆與場站標記說明表.csv'
)

label_table.to_csv(
    LABEL_FILE,
    index=False,
    encoding='utf-8-sig',
)


# ==================== J. 完整驗證 ====================

if source_total_rows != output_total_rows:
    raise RuntimeError(
        '原始資料與輸出資料筆數不一致：\n'
        f'原始總筆數：{source_total_rows:,}\n'
        f'輸出總筆數：{output_total_rows:,}'
    )

print('\n各月份筆數：')

for month in range(1, 7):
    output_path = monthly_paths[month]

    if not output_path.exists():
        raise FileNotFoundError(
            f'{month:02d}月沒有產生輸出檔。'
        )

    # 重新讀取表頭，確認欄位及順序
    output_header = pd.read_csv(
        output_path,
        encoding='utf-8-sig',
        nrows=0,
    )

    actual_columns = list(
        output_header.columns
    )

    if actual_columns != FINAL_COLUMNS:
        raise RuntimeError(
            f'{month:02d}月欄位順序錯誤：\n'
            f'實際：{actual_columns}\n'
            f'應為：{FINAL_COLUMNS}'
        )

    print(
        f'{month:02d}月：'
        f'{month_counts[month]:,}筆'
    )

print('\n分鐘分布：')

for minute in sorted(minute_counts):
    print(
        f'{minute:02d}分：'
        f'{minute_counts[minute]:,}筆'
    )

print('\n資料驗證完成：')
print(f'原始總筆數：{source_total_rows:,}')
print(f'輸出總筆數：{output_total_rows:,}')
print(f'300公尺內無指定類型筆數：{unclassified_rows:,}')
print(f'300公尺內無指定類型場站數：{len(unclassified_stations):,}')

if unclassified_stations:
    print(
        '\n以下場站不是合併失敗，'
        '而是300公尺內沒有七種指定設施：'
    )

    display(
        pd.DataFrame({
            '場站名稱': sorted(
                unclassified_stations
            )
        }).head(30)
    )


# ==================== K. 打包六個月＋說明表 ====================

output_files = [
    monthly_paths[month]
    for month in range(1, 7)
]

output_files.append(
    LABEL_FILE
)

if len(output_files) != 7:
    raise RuntimeError(
        f'應有7個CSV，目前為'
        f'{len(output_files)}個。'
    )

ZIP_OUTPUT = (
    Path('/content') /
    'training_data_1至6月_完整時段_'
    '日曆_場站_中文版_共7檔.zip'
)

if ZIP_OUTPUT.exists():
    ZIP_OUTPUT.unlink()

with zipfile.ZipFile(
    ZIP_OUTPUT,
    mode='w',
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zf:
    for output_file in output_files:
        zf.write(
            output_file,
            arcname=output_file.name,
        )


# 驗證ZIP內容
with zipfile.ZipFile(
    ZIP_OUTPUT,
    mode='r',
) as zf:
    zip_names = zf.namelist()

if len(zip_names) != 7:
    raise RuntimeError(
        f'ZIP內應有7個CSV，'
        f'實際為{len(zip_names)}個。'
    )

print('\n開始下載到本地端……')

files.download(
    str(ZIP_OUTPUT)
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
原始完整資料： /content/drive/MyDrive/Hackathon_Data/training data_一至六月資料.zip
場站分類對照： /content/drive/MyDrive/Hackathon_Data/YouBike場站300公尺類型對照.csv

找到6個原始CSV：
 - training data_一至六月資料/dataset_April.csv
 - training data_一至六月資料/dataset_February.csv
 - training data_一至六月資料/dataset_January.csv
 - training data_一至六月資料/dataset_June.csv
 - training data_一至六月資料/dataset_March.csv
 - training data_一至六月資料/dataset_May.csv

成功載入場站對照：1,577站
有場站分類： 1,569站
300公尺內無指定類型： 8站


,城市,行政區,場站名稱,場站
0,新北市,三峽區,三峽中山公園,"公車站,學校,公園,醫院"
1,新北市,三峽區,三峽區公所(三峽老街),"公車站,學校,公園,商圈"
2,新北市,三峽區,三峽區礁溪行政中心,"公車站,學校,公園"
3,新北市,三峽區,三峽原住民族生活文化園區,"學校,公園"
4,新北市,三峽區,三峽國中,"捷運站,公車站,學校,公園"
5,新北市,三峽區,三峽國光青年社會住宅,"捷運站,公車站,學校,公園"
6,新北市,三峽區,三峽國小(中山路),"公車站,學校,公園,商圈"
7,新北市,三峽區,三樹大學路口,"捷運站,公車站,學校,公園"
8,新北市,三峽區,三樹大德路口,"公車站,學校,公園,商圈"
9,新北市,三峽區,三鶯國民運動中心,"捷運站,公車站,學校,公園,商圈"



[1/6] 處理：dataset_April.csv
  第1批完成：300,000筆
  第2批完成：300,000筆
  第3批完成：300,000筆
  第4批完成：300,000筆
  第5批完成：300,000筆
  第6批完成：300,000筆
  第7批完成：300,000筆
  第8批完成：110,968筆

[2/6] 處理：dataset_February.csv
  第1批完成：300,000筆
  第2批完成：300,000筆
  第3批完成：300,000筆
  第4批完成：300,000筆
  第5批完成：300,000筆
  第6批完成：300,000筆
  第7批完成：247,641筆

[3/6] 處理：dataset_January.csv
  第1批完成：300,000筆
  第2批完成：300,000筆
  第3批完成：300,000筆
  第4批完成：300,000筆
  第5批完成：300,000筆
  第6批完成：300,000筆
  第7批完成：300,000筆
  第8批完成：154,674筆

[4/6] 處理：dataset_June.csv
  第1批完成：300,000筆
  第2批完成：300,000筆
  第3批完成：300,000筆
  第4批完成：300,000筆
  第5批完成：300,000筆
  第6批完成：300,000筆
  第7批完成：300,000筆
  第8批完成：140,462筆

[5/6] 處理：dataset_March.csv
  第1批完成：300,000筆
  第2批完成：300,000筆
  第3批完成：300,000筆
  第4批完成：300,000筆
  第5批完成：300,000筆
  第6批完成：300,000筆
  第7批完成：300,000筆
  第8批完成：174,046筆

[6/6] 處理：dataset_May.csv
  第1批完成：300,000筆
  第2批完成：300,000筆
  第3批完成：300,000筆
  第4批完成：300,000筆
  第5批完成：300,000筆
  第6批完成：300,000筆
  第7批完成：300,000筆
  第8批完成：197,154筆

各月份筆數：
01月：2,254,674筆
02月：2,04

,場站名稱
0,三貂嶺隧道(金字橋)
1,中和慈惠堂
2,和平路105號
3,東大街
4,東豐街85巷口
5,柑園街二段363巷口
6,樹林區第三平面停車場
7,直潭淨水廠(直潭二街)



開始下載到本地端……


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>